# World RL Production/Paper-Trading Notebook

Selected model: `v12b_small_rl_guardian`

This cleaned copy removes the old research branches and keeps only the production-relevant path:

1. Setup, Drive mount, and optional Massive API smoke test.
2. Load cached V6/V12B/RL Guardian artifacts from Drive.
3. Validate exact same-leg inverse live-order mapping.
4. Cache-load or rebuild the small RL Guardian sizing overlay.
5. Export the full backtest package in the reference 8-CSV format.
6. Export the full model artifact with live state and metadata.
7. Print the final model performance summary.

Live trading is intentionally disabled in the artifact until paper-trading and broker order validation pass.


In [ ]:
# Install only the production/paper-trading dependencies
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "requests": "requests",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in REQUIREMENTS.items():
    ensure_package(import_name, pip_name)

print("Production dependencies ready.")


In [ ]:
# Imports, production config, and fixed model identity
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from getpass import getpass
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)

@dataclass
class ProductionConfig:
    data_root: str = "/content/drive/MyDrive/quant_event_vol_world_model_v2"
    output_subfolder: str = "outputs_brppo_guardian"

    massive_api_key: str = ""
    request_sleep_seconds: float = 0.12

    cash_rate: float = 0.045
    annualization_days: int = 252
    initial_capital: float = 100000.0
    random_seed: int = 42

    # Kept for compatibility with cached option/order helpers.
    option_min_dte: int = 10
    option_max_dte: int = 75
    option_strike_band: float = 0.25

cfg = ProductionConfig()

env_key = os.environ.get("MASSIVE_API_KEY", "").strip() or os.environ.get("POLYGON_API_KEY", "").strip()
if env_key:
    cfg.massive_api_key = env_key

np.random.seed(cfg.random_seed)
random.seed(cfg.random_seed)

DATA_ROOT = Path(cfg.data_root)
OUTPUT_ROOT = DATA_ROOT / cfg.output_subfolder
V6_ROOT = OUTPUT_ROOT / "brppo_guardian_execution"
V6_LOG_ROOT = V6_ROOT / "logs"
V6_CACHE_ROOT = V6_ROOT / "cache"
V12B_ROOT = V6_ROOT / "v12b_exact_same_leg_inverse"
V12B_LIVE_ROOT = V12B_ROOT / "live_deployment_mapping"

BEST_MODEL_NAME = "v12b_small_rl_guardian"
BEST_MODEL_VERSION = "v12b_exact_same_leg_inverse_plus_small_rl_guardian_v1"
BEST_STRATEGY_NAME = "V12B Exact Same-Leg Inverse + Small RL Guardian"
LIVE_TRADING_ENABLED = False

FROZEN = {
    "cash_rate": cfg.cash_rate,
    "max_scale": 3,
    "v12b_inverse_scale": 2.0,
    "live_trading_enabled": LIVE_TRADING_ENABLED,
}

def v5_cash(index):
    idx = pd.to_datetime(index).normalize()
    return pd.Series((1.0 + cfg.cash_rate) ** (1.0 / cfg.annualization_days) - 1.0, index=idx).fillna(0.0)

print("Data root:", DATA_ROOT)
print("Selected production model:", BEST_MODEL_NAME)
print("Live trading enabled:", LIVE_TRADING_ENABLED)
display(pd.Series(asdict(cfg)).to_frame("value"))


In [ ]:
# Mount Google Drive and load Massive API key
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
PROMPT_FOR_MASSIVE_API_KEY = False #@param {type:"boolean"}

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped or already mounted: {exc}")

for p in [DATA_ROOT, OUTPUT_ROOT, V6_ROOT, V6_LOG_ROOT, V6_CACHE_ROOT, V12B_ROOT, V12B_LIVE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

key = (
    os.environ.get("MASSIVE_API_KEY", "").strip()
    or os.environ.get("POLYGON_API_KEY", "").strip()
    or getattr(cfg, "massive_api_key", "")
)

if not key:
    try:
        from google.colab import userdata
        key = (userdata.get("MASSIVE_API_KEY") or "").strip()
    except Exception:
        key = ""

if PROMPT_FOR_MASSIVE_API_KEY and not key:
    key = getpass("Paste Massive API key: ").strip()

cfg.massive_api_key = key
if key:
    os.environ["MASSIVE_API_KEY"] = key

print("Massive API key loaded for this runtime:", bool(cfg.massive_api_key))


In [ ]:
# Optional Massive API smoke test
RUN_API_SMOKE_TEST = False #@param {type:"boolean"}

def smoke_request(name: str, path: str, params: dict) -> dict:
    if not cfg.massive_api_key:
        return {"test": name, "status_code": np.nan, "ok": False, "message": "Missing Massive API key."}
    url = "https://api.massive.com" + path
    clean = dict(params)
    clean["apiKey"] = cfg.massive_api_key
    try:
        r = requests.get(url, params=clean, headers={"accept": "application/json", "connection": "close"}, timeout=(10, 60))
        txt = r.text[:500]
        ok = 200 <= r.status_code < 300
        return {"test": name, "status_code": r.status_code, "ok": ok, "message": txt}
    except Exception as exc:
        return {"test": name, "status_code": np.nan, "ok": False, "message": repr(exc)}

if RUN_API_SMOKE_TEST:
    today = pd.Timestamp.today().normalize()
    smoke_rows = []
    smoke_rows.append(smoke_request("Benzinga earnings", "/benzinga/v1/earnings", {
        "date.gte": (today - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
        "date.lte": today.strftime("%Y-%m-%d"),
        "limit": 1,
    }))
    smoke_rows.append(smoke_request("Options contracts reference", "/v3/reference/options/contracts", {
        "underlying_ticker": "AAPL",
        "as_of": today.strftime("%Y-%m-%d"),
        "expired": "false",
        "expiration_date.gte": (today + pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
        "expiration_date.lte": (today + pd.Timedelta(days=75)).strftime("%Y-%m-%d"),
        "limit": 1,
    }))
    smoke_df = pd.DataFrame(smoke_rows)
    display(smoke_df)
    if not smoke_df["ok"].all():
        raise RuntimeError("API smoke test failed. Fix entitlement/key before paper/live order generation.")
else:
    print("API smoke test skipped.")


In [ ]:
# Cached artifact helpers and core metrics
EXPECTED_REFERENCE_EXPORT_FILES = [
    "all_periods_all_models_metrics.csv",
    "all_periods_all_models_daily_backtest.csv",
    "all_models_signal_level.csv",
    "all_periods_all_models_trade_log.csv",
    "all_periods_all_models_order_log.csv",
    "all_periods_all_models_position_log.csv",
    "all_periods_all_models_drawdown_log.csv",
    "backtest_export_errors.csv",
]

def first_existing(paths):
    for p in [Path(x) for x in paths]:
        if p.exists():
            return p
    return None

def load_series_csv(path, name):
    path = Path(path)
    df = pd.read_csv(path)
    date_col = "date" if "date" in df.columns else df.columns[0]
    value_candidates = [name, "daily_return", "return", "ret", "portfolio_ret"]
    val_col = next((c for c in value_candidates if c in df.columns and c != date_col), None)
    if val_col is None:
        numeric_cols = [c for c in df.columns if c != date_col and pd.api.types.is_numeric_dtype(df[c])]
        val_col = numeric_cols[0] if numeric_cols else [c for c in df.columns if c != date_col][0]
    s = pd.Series(pd.to_numeric(df[val_col], errors="coerce").fillna(0.0).values,
                  index=pd.to_datetime(df[date_col], errors="coerce").dt.normalize(),
                  name=name)
    s = s[~pd.isna(s.index)]
    return s.groupby(level=0).sum().sort_index().rename(name)

def ann_metrics(r, name):
    r = pd.Series(r).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
    if r.empty:
        return {"name": name, "annual_return": np.nan, "annual_vol": np.nan, "sharpe": np.nan,
                "max_drawdown": np.nan, "calmar": np.nan, "worst_day": np.nan,
                "final_equity": np.nan, "n_days": 0}
    eq = (1.0 + r).cumprod()
    years = max(len(r) / cfg.annualization_days, 1 / cfg.annualization_days)
    final_equity = float(eq.iloc[-1])
    ann = final_equity ** (1.0 / years) - 1.0 if final_equity > 0 else -1.0
    vol = float(r.std(ddof=1) * np.sqrt(cfg.annualization_days)) if len(r) > 1 else 0.0
    sharpe = float(np.sqrt(cfg.annualization_days) * r.mean() / r.std(ddof=1)) if len(r) > 1 and r.std(ddof=1) > 1e-12 else np.nan
    dd = 1.0 - eq / eq.cummax()
    max_dd = float(dd.max())
    return {
        "name": name,
        "annual_return": ann,
        "annual_vol": vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "calmar": ann / max_dd if max_dd > 1e-12 else np.nan,
        "worst_day": float(r.min()),
        "final_equity": final_equity,
        "n_days": int(len(r)),
    }

def file_status(paths):
    rows = []
    for label, path in paths.items():
        p = Path(path)
        rows.append({"artifact": label, "path": str(p), "exists": p.exists(), "size_bytes": p.stat().st_size if p.exists() else 0})
    return pd.DataFrame(rows)


In [ ]:
# Load cached V6, V12B, and RL Guardian outputs
v6_real_paths_path = first_existing([
    V6_CACHE_ROOT / "v6_real_option_structure_paths.parquet",
    V6_LOG_ROOT / "v6_real_option_structure_paths.parquet",
])
v6_real_daily_path = first_existing([
    V6_LOG_ROOT / "v6_real_ironfly_daily_returns.csv",
])
v6_frozen_decisions_path = first_existing([
    V6_LOG_ROOT / "v6_frozen_decisions.csv",
])

v12b_raw_daily_path = first_existing([
    V12B_ROOT / "v12b_best_model_daily_returns.csv",
])
v12b_summary_path = first_existing([V12B_ROOT / "v12b_summary.csv"])
v12b_yearly_path = first_existing([V12B_ROOT / "v12b_yearly_metrics.csv"])

rl_daily_path = first_existing([
    V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_daily_returns.csv",
])
rl_actions_path = first_existing([
    V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_actions.csv",
])
rl_summary_path = first_existing([
    V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_summary.csv",
])

display(file_status({
    "v6_real_paths": v6_real_paths_path or V6_CACHE_ROOT / "v6_real_option_structure_paths.parquet",
    "v6_real_daily": v6_real_daily_path or V6_LOG_ROOT / "v6_real_ironfly_daily_returns.csv",
    "v6_frozen_decisions": v6_frozen_decisions_path or V6_LOG_ROOT / "v6_frozen_decisions.csv",
    "v12b_raw_daily": v12b_raw_daily_path or V12B_ROOT / "v12b_best_model_daily_returns.csv",
    "v12b_summary": v12b_summary_path or V12B_ROOT / "v12b_summary.csv",
    "v12b_yearly": v12b_yearly_path or V12B_ROOT / "v12b_yearly_metrics.csv",
    "rl_guardian_daily": rl_daily_path or V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_daily_returns.csv",
    "rl_guardian_actions": rl_actions_path or V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_actions.csv",
    "rl_guardian_summary": rl_summary_path or V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_summary.csv",
}))

v6_real_paths = pd.read_parquet(v6_real_paths_path) if v6_real_paths_path else pd.DataFrame()
v6_real_daily = load_series_csv(v6_real_daily_path, "v6_real_ironfly_daily") if v6_real_daily_path else pd.Series(dtype=float)
v6_frozen_decisions = pd.read_csv(v6_frozen_decisions_path) if v6_frozen_decisions_path else pd.DataFrame()

if v12b_raw_daily_path is None:
    raise RuntimeError("Missing V12B raw daily cache. Expected v12b_best_model_daily_returns.csv under V12B_ROOT.")

V12B_RAW_DAILY = load_series_csv(v12b_raw_daily_path, "v12b_exact_same_leg_inverse_raw")
V12B_SUMMARY = pd.read_csv(v12b_summary_path) if v12b_summary_path else pd.DataFrame([ann_metrics(V12B_RAW_DAILY, "v12b_exact_same_leg_inverse_raw")])
V12B_YEARLY = pd.read_csv(v12b_yearly_path) if v12b_yearly_path else pd.DataFrame()

if rl_daily_path:
    V12B_RL_GUARDIAN_DAILY = load_series_csv(rl_daily_path, "v12b_small_rl_guardian")
    BEST_MODEL_RETURNS = V12B_RL_GUARDIAN_DAILY.copy()
else:
    BEST_MODEL_RETURNS = pd.Series(dtype=float, name="v12b_small_rl_guardian")

V12B_RL_GUARDIAN_ACTIONS = pd.read_csv(rl_actions_path) if rl_actions_path else pd.DataFrame()
V12B_RL_GUARDIAN_SUMMARY = pd.read_csv(rl_summary_path) if rl_summary_path else pd.DataFrame()

BEST_MODEL_NAME = "v12b_small_rl_guardian"
BEST_MODEL_RETURNS.name = BEST_MODEL_NAME

display(V12B_SUMMARY)
if not V12B_RL_GUARDIAN_SUMMARY.empty:
    display(V12B_RL_GUARDIAN_SUMMARY)
print("Loaded V12B raw days:", len(V12B_RAW_DAILY))
print("Loaded RL Guardian days:", len(BEST_MODEL_RETURNS))
print("Active BEST_MODEL:", BEST_MODEL_NAME)


In [ ]:
# Build or load exact same-leg inverse live mapping
def normalize_dates(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
    return out

live_decisions_path = V12B_LIVE_ROOT / "v12b_live_selected_decisions.csv"
live_order_map_path = V12B_LIVE_ROOT / "v12b_live_order_map.csv"

if live_decisions_path.exists():
    V12B_SELECTED_DECISIONS = pd.read_csv(live_decisions_path)
    V12B_SELECTED_DECISIONS = normalize_dates(V12B_SELECTED_DECISIONS, ["entry_date", "event_date", "exit_date"])
else:
    base_decisions = v6_frozen_decisions.copy() if isinstance(v6_frozen_decisions, pd.DataFrame) else pd.DataFrame()
    paths_for_keys = v6_real_paths.copy() if isinstance(v6_real_paths, pd.DataFrame) else pd.DataFrame()
    if base_decisions.empty and not paths_for_keys.empty:
        cols = [c for c in ["ticker", "entry_date", "action", "structure"] if c in paths_for_keys.columns]
        base_decisions = paths_for_keys[cols].drop_duplicates().copy()

    if not base_decisions.empty and not paths_for_keys.empty:
        base_decisions["ticker"] = base_decisions["ticker"].astype(str).str.upper()
        base_decisions["entry_date"] = pd.to_datetime(base_decisions["entry_date"], errors="coerce").dt.normalize()
        base_decisions["action"] = base_decisions["action"].astype(str)
        paths_for_keys["ticker"] = paths_for_keys["ticker"].astype(str).str.upper()
        paths_for_keys["entry_date"] = pd.to_datetime(paths_for_keys["entry_date"], errors="coerce").dt.normalize()
        paths_for_keys["action"] = paths_for_keys["action"].astype(str)
        active_keys = paths_for_keys[["ticker", "entry_date", "action"]].drop_duplicates()
        V12B_SELECTED_DECISIONS = base_decisions.merge(active_keys, on=["ticker", "entry_date", "action"], how="inner")
    else:
        V12B_SELECTED_DECISIONS = pd.DataFrame()

    if not V12B_SELECTED_DECISIONS.empty:
        V12B_SELECTED_DECISIONS["original_model_action"] = V12B_SELECTED_DECISIONS["action"]
        V12B_SELECTED_DECISIONS["execution_mode"] = "exact_same_leg_inverse"
        V12B_SELECTED_DECISIONS["live_execution_action"] = V12B_SELECTED_DECISIONS["action"].map({
            "short_vol_defined": "take_opposite_side_of_same_iron_fly",
            "long_vol": "take_opposite_side_of_same_straddle",
        }).fillna("do_not_trade_unknown_action")
        V12B_SELECTED_DECISIONS["selected_model"] = BEST_MODEL_NAME
        V12B_SELECTED_DECISIONS["selected_model_version"] = BEST_MODEL_VERSION
        V12B_SELECTED_DECISIONS.to_csv(live_decisions_path, index=False)

if live_order_map_path.exists():
    V12B_LIVE_ORDER_MAP = pd.read_csv(live_order_map_path)
else:
    paths_for_map = v6_real_paths.copy() if isinstance(v6_real_paths, pd.DataFrame) else pd.DataFrame()
    order_rows = []
    exit_side = {"BUY_TO_OPEN": "SELL_TO_CLOSE", "SELL_TO_OPEN": "BUY_TO_CLOSE"}

    def add_leg(base, role, option_ticker, side):
        if option_ticker is None or pd.isna(option_ticker) or str(option_ticker) in ["", "None", "nan"]:
            return
        row = dict(base)
        row.update({
            "leg_role": role,
            "option_ticker": str(option_ticker),
            "quantity": 1,
            "entry_order_side": side,
            "exit_order_side": exit_side[side],
            "selected_model": BEST_MODEL_NAME,
            "selected_model_version": BEST_MODEL_VERSION,
        })
        order_rows.append(row)

    if not paths_for_map.empty:
        paths_for_map["ticker"] = paths_for_map["ticker"].astype(str).str.upper()
        paths_for_map["entry_date"] = pd.to_datetime(paths_for_map["entry_date"], errors="coerce").dt.normalize()
        paths_for_map["action"] = paths_for_map["action"].astype(str)
        trade_cols = ["ticker", "entry_date", "action", "structure", "short_call", "short_put", "long_call_wing", "long_put_wing"]
        trades = paths_for_map[[c for c in trade_cols if c in paths_for_map.columns]].drop_duplicates(["ticker", "entry_date", "action"])
        for _, tr in trades.iterrows():
            base = {
                "ticker": tr["ticker"],
                "entry_date": pd.Timestamp(tr["entry_date"]).strftime("%Y-%m-%d"),
                "original_model_action": tr["action"],
                "original_structure": tr.get("structure", ""),
                "execution_mode": "exact_same_leg_inverse",
            }
            if tr["action"] == "short_vol_defined":
                base["live_execution_action"] = "take_opposite_side_of_same_iron_fly"
                add_leg(base, "original_short_call", tr.get("short_call"), "BUY_TO_OPEN")
                add_leg(base, "original_short_put", tr.get("short_put"), "BUY_TO_OPEN")
                add_leg(base, "original_long_call_wing", tr.get("long_call_wing"), "SELL_TO_OPEN")
                add_leg(base, "original_long_put_wing", tr.get("long_put_wing"), "SELL_TO_OPEN")
            elif tr["action"] == "long_vol":
                base["live_execution_action"] = "take_opposite_side_of_same_straddle"
                add_leg(base, "original_call", tr.get("short_call"), "SELL_TO_OPEN")
                add_leg(base, "original_put", tr.get("short_put"), "SELL_TO_OPEN")

    V12B_LIVE_ORDER_MAP = pd.DataFrame(order_rows)
    V12B_LIVE_ORDER_MAP.to_csv(live_order_map_path, index=False)

BEST_MODEL_SELECTED_DECISIONS = V12B_SELECTED_DECISIONS.copy()

if not V12B_SELECTED_DECISIONS.empty:
    display(V12B_SELECTED_DECISIONS.head())
else:
    print("Warning: selected decisions are empty. Artifact will still export, but live map needs cached V6/V12B paths.")

if not V12B_LIVE_ORDER_MAP.empty:
    display(V12B_LIVE_ORDER_MAP.head(20))
else:
    print("Warning: live order map is empty. Load v6_real_paths or rerun the V12B live mapping step before paper trading.")

print("Live mapping folder:", V12B_LIVE_ROOT)


In [ ]:
# Cell 2 - Small RL Risk Guardian sizing overlay for V12B

RUN_SMALL_RL_RISK_GUARDIAN = True #@param {type:"boolean"}
RL_GUARDIAN_EPISODES = 160 #@param {type:"integer"}
RL_GUARDIAN_DD_BUDGET = 0.18 #@param {type:"number"}
RL_GUARDIAN_WORST_DAY_BUDGET = 0.10 #@param {type:"number"}
RL_GUARDIAN_SELECT_IF_DD_IMPROVES = True #@param {type:"boolean"}

REUSE_SMALL_RL_GUARDIAN_CACHE = True #@param {type:"boolean"}
FORCE_RL_GUARDIAN_AS_BEST_MODEL = True

_RL_OUT = V12B_ROOT / "small_rl_guardian"
_RL_DAILY_PATH = _RL_OUT / "v12b_small_rl_guardian_daily_returns.csv"
_RL_ACTIONS_PATH = _RL_OUT / "v12b_small_rl_guardian_actions.csv"
_RL_SUMMARY_PATH = _RL_OUT / "v12b_small_rl_guardian_summary.csv"

if RUN_SMALL_RL_RISK_GUARDIAN and REUSE_SMALL_RL_GUARDIAN_CACHE and _RL_DAILY_PATH.exists() and _RL_ACTIONS_PATH.exists() and _RL_SUMMARY_PATH.exists():
    V12B_RL_GUARDIAN_DAILY = load_series_csv(_RL_DAILY_PATH, "v12b_small_rl_guardian")
    V12B_RL_GUARDIAN_ACTIONS = pd.read_csv(_RL_ACTIONS_PATH)
    V12B_RL_GUARDIAN_SUMMARY = pd.read_csv(_RL_SUMMARY_PATH)

    BEST_MODEL_NAME = "v12b_small_rl_guardian"
    BEST_MODEL_RETURNS = V12B_RL_GUARDIAN_DAILY.copy()
    BEST_MODEL_SUMMARY = V12B_RL_GUARDIAN_SUMMARY.copy()
    BEST_MODEL_SELECTED_DECISIONS = V12B_SELECTED_DECISIONS.copy() if "V12B_SELECTED_DECISIONS" in globals() else pd.DataFrame()

    display(V12B_RL_GUARDIAN_SUMMARY)
    print("Loaded cached RL Guardian model as BEST_MODEL:", BEST_MODEL_NAME)
elif RUN_SMALL_RL_RISK_GUARDIAN:
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from collections import defaultdict

    raw = pd.Series(V12B_RAW_DAILY).copy()
    raw.index = pd.to_datetime(raw.index).normalize()
    raw = pd.to_numeric(raw, errors="coerce").fillna(0.0).sort_index()

    def cash_series(index):
        idx = pd.to_datetime(index).normalize()
        if "v5_cash" in globals():
            return pd.Series(v5_cash(idx), index=idx).fillna(0.0)
        rate = float(FROZEN.get("cash_rate", 0.045)) if "FROZEN" in globals() else 0.045
        return pd.Series((1 + rate) ** (1 / 252) - 1, index=idx)

    def metric(s, name):
        s = pd.Series(s).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
        eq = (1 + s).cumprod()
        years = max(len(s) / 252, 1 / 252)
        final_eq = float(eq.iloc[-1])
        ann = final_eq ** (1 / years) - 1 if final_eq > 0 else -1
        vol = float(s.std(ddof=1) * np.sqrt(252)) if len(s) > 1 else 0
        sharpe = float(np.sqrt(252) * s.mean() / s.std(ddof=1)) if len(s) > 1 and s.std(ddof=1) > 1e-12 else np.nan
        dd = 1 - eq / eq.cummax()
        return {"name": name, "annual_return": ann, "annual_vol": vol, "sharpe": sharpe,
                "max_drawdown": float(dd.max()), "worst_day": float(s.min()),
                "final_equity": final_eq, "n_days": int(len(s))}

    def add_excess(row, daily):
        cash = cash_series(daily.index)
        ex = pd.Series(daily).reindex(cash.index).fillna(0.0) - cash
        row["excess_annual_return"] = float(ex.mean() * 252)
        row["excess_annual_vol"] = float(ex.std(ddof=1) * np.sqrt(252)) if len(ex) > 1 else np.nan
        row["excess_sharpe"] = float(np.sqrt(252) * ex.mean() / ex.std(ddof=1)) if len(ex) > 1 and ex.std(ddof=1) > 1e-12 else np.nan
        return row

    cash = cash_series(raw.index)
    alpha = raw - cash

    if "v12b_positions" in globals() and isinstance(v12b_positions, pd.DataFrame) and not v12b_positions.empty:
        pos = v12b_positions.copy()
    elif (V12B_ROOT / "v12b_positions.parquet").exists():
        pos = pd.read_parquet(V12B_ROOT / "v12b_positions.parquet")
    else:
        pos = pd.DataFrame()

    if not pos.empty and "path_date" in pos.columns and "margin_required_pct" in pos.columns:
        pos["path_date"] = pd.to_datetime(pos["path_date"], errors="coerce").dt.normalize()
        margin_daily = pos.groupby("path_date")["margin_required_pct"].sum().reindex(raw.index).fillna(0.0)
    else:
        margin_daily = pd.Series(0.0, index=raw.index)

    actions_grid = np.array([0.00, 0.25, 0.50, 0.75, 1.00], dtype=float)

    def bucket(x, edges):
        return int(np.searchsorted(edges, float(x), side="right"))

    def state_at(t, eq, high, last_mult, a_arr, m_arr):
        dd = 1 - eq / max(high, 1e-12)
        past20 = a_arr[max(0, t-20):t]
        past5 = a_arr[max(0, t-5):t]
        vol20 = np.std(past20, ddof=1) * np.sqrt(252) if len(past20) > 2 else 0.0
        mom5 = np.sum(past5) if len(past5) else 0.0
        return (
            bucket(dd, [0.03, 0.07, 0.12, 0.18]),
            bucket(vol20, [0.15, 0.30, 0.50, 0.80]),
            bucket(mom5, [-0.05, 0.00, 0.05, 0.15]),
            bucket(m_arr[t] if t < len(m_arr) else 0.0, [0.20, 0.50, 0.80, 1.20]),
            bucket(last_mult, [0.01, 0.30, 0.60, 0.90]),
        )

    def train_q(alpha_train, cash_train, margin_train, seed):
        rng = np.random.default_rng(seed)
        a_arr = np.asarray(alpha_train, dtype=float)
        c_arr = np.asarray(cash_train, dtype=float)
        m_arr = np.asarray(margin_train, dtype=float)
        q = defaultdict(lambda: np.zeros(len(actions_grid), dtype=float))
        gamma, lr0 = 0.94, 0.10

        for ep in range(int(RL_GUARDIAN_EPISODES)):
            eq, high, last_mult = 1.0, 1.0, 1.0
            eps = max(0.03, 0.35 * (1 - ep / max(int(RL_GUARDIAN_EPISODES), 1)))
            lr = lr0 * (0.5 + 0.5 * (1 - ep / max(int(RL_GUARDIAN_EPISODES), 1)))

            for t in range(len(a_arr)):
                st = state_at(t, eq, high, last_mult, a_arr, m_arr)
                if rng.random() < eps:
                    ai = int(rng.integers(0, len(actions_grid)))
                else:
                    ai = int(np.argmax(q[st]))

                mult = float(actions_grid[ai])
                ret = float(c_arr[t] + mult * a_arr[t])
                turnover = abs(mult - last_mult)

                next_eq = eq * max(0.001, 1 + ret)
                next_high = max(high, next_eq)
                dd = 1 - next_eq / max(next_high, 1e-12)
                margin_used = mult * float(m_arr[t])

                reward = 24.0 * (mult * a_arr[t]) + 4.0 * ret
                reward -= 20.0 * max(dd - float(RL_GUARDIAN_DD_BUDGET), 0.0) ** 2
                reward -= 14.0 * max(-ret - float(RL_GUARDIAN_WORST_DAY_BUDGET), 0.0) ** 2
                reward -= 0.04 * turnover
                reward -= 1.50 * max(margin_used - 0.90, 0.0)

                next_st = state_at(min(t + 1, len(a_arr) - 1), next_eq, next_high, mult, a_arr, m_arr)
                q[st][ai] += lr * (reward + gamma * np.max(q[next_st]) - q[st][ai])

                eq, high, last_mult = next_eq, next_high, mult
        return q

    def apply_q(alpha_test, cash_test, margin_test, q, idx):
        a_arr = np.asarray(alpha_test, dtype=float)
        c_arr = np.asarray(cash_test, dtype=float)
        m_arr = np.asarray(margin_test, dtype=float)
        eq, high, last_mult = 1.0, 1.0, 1.0
        rows = []

        for t, dt in enumerate(idx):
            st = state_at(t, eq, high, last_mult, a_arr, m_arr)
            ai = int(np.argmax(q[st])) if st in q else len(actions_grid) - 1
            mult = float(actions_grid[ai])
            ret = float(c_arr[t] + mult * a_arr[t])
            eq *= max(0.001, 1 + ret)
            high = max(high, eq)
            rows.append({"date": dt, "rl_multiplier": mult, "raw_alpha": a_arr[t],
                         "cash_return": c_arr[t], "guarded_return": ret,
                         "drawdown_state": 1 - eq / max(high, 1e-12),
                         "margin_proxy": float(m_arr[t])})
            last_mult = mult

        out = pd.DataFrame(rows)
        s = pd.Series(out["guarded_return"].values, index=pd.to_datetime(out["date"]).dt.normalize()).rename("v12b_small_rl_guardian")
        return s, out

    parts, action_parts = [], []
    for y in sorted(raw.index.year.unique()):
        test_idx = raw.index[raw.index.year == y]
        train_idx = raw.index[raw.index.year < y]

        if len(train_idx) < 120:
            guarded = raw.reindex(test_idx).copy().rename("v12b_small_rl_guardian")
            acts = pd.DataFrame({"date": test_idx, "rl_multiplier": 1.0, "guarded_return": guarded.values, "note": "insufficient_prior_training"})
        else:
            q = train_q(alpha.reindex(train_idx), cash.reindex(train_idx), margin_daily.reindex(train_idx), seed=31000 + int(y))
            guarded, acts = apply_q(alpha.reindex(test_idx), cash.reindex(test_idx), margin_daily.reindex(test_idx), q, test_idx)

        acts["fold_test_year"] = int(y)
        parts.append(guarded)
        action_parts.append(acts)

    V12B_RL_GUARDIAN_DAILY = pd.concat(parts).sort_index().rename("v12b_small_rl_guardian")
    V12B_RL_GUARDIAN_ACTIONS = pd.concat(action_parts, ignore_index=True)

    raw_row = add_excess(metric(raw, "v12b_raw"), raw)
    guard_row = add_excess(metric(V12B_RL_GUARDIAN_DAILY, "v12b_small_rl_guardian"), V12B_RL_GUARDIAN_DAILY)
    V12B_RL_GUARDIAN_SUMMARY = pd.DataFrame([raw_row, guard_row])
    display(V12B_RL_GUARDIAN_SUMMARY)

    raw_score = raw_row["annual_return"] + 0.25 * np.nan_to_num(raw_row["sharpe"], nan=0) - 1.5 * raw_row["max_drawdown"]
    guard_score = guard_row["annual_return"] + 0.25 * np.nan_to_num(guard_row["sharpe"], nan=0) - 1.5 * guard_row["max_drawdown"]

    guardian_target_ok = (
        guard_row["annual_return"] >= 0.30
        and guard_row["excess_annual_return"] >= 0.25
        and guard_row["max_drawdown"] <= 0.25
    )
    dd_improved = guard_row["max_drawdown"] < raw_row["max_drawdown"] * 0.95

    if guardian_target_ok and ((RL_GUARDIAN_SELECT_IF_DD_IMPROVES and dd_improved) or guard_score > raw_score):
        BEST_MODEL_NAME = "v12b_small_rl_guardian"
        BEST_MODEL_RETURNS = V12B_RL_GUARDIAN_DAILY.copy()
        print("Selected RL Guardian model as BEST_MODEL.")
    else:
        BEST_MODEL_NAME = "v12b_exact_same_leg_inverse_raw"
        BEST_MODEL_RETURNS = raw.copy()
        print("Kept raw V12B as BEST_MODEL. RL Guardian saved as comparison.")

    BEST_MODEL_SUMMARY = V12B_RL_GUARDIAN_SUMMARY.copy()
    BEST_MODEL_SELECTED_DECISIONS = V12B_SELECTED_DECISIONS.copy()

    OUT = V12B_ROOT / "small_rl_guardian"
    OUT.mkdir(parents=True, exist_ok=True)
    V12B_RL_GUARDIAN_DAILY.to_csv(OUT / "v12b_small_rl_guardian_daily_returns.csv")
    V12B_RL_GUARDIAN_ACTIONS.to_csv(OUT / "v12b_small_rl_guardian_actions.csv", index=False)
    V12B_RL_GUARDIAN_SUMMARY.to_csv(OUT / "v12b_small_rl_guardian_summary.csv", index=False)

    eq = (1 + BEST_MODEL_RETURNS).cumprod()
    dd = 1 - eq / eq.cummax()
    eq.plot(figsize=(14, 4), title=f"{BEST_MODEL_NAME} Equity")
    plt.show()
    dd.plot(figsize=(14, 3), title=f"{BEST_MODEL_NAME} Drawdown")
    plt.show()

    print("Active BEST_MODEL:", BEST_MODEL_NAME)

# Production lock: all downstream cells must use the RL Guardian variant.
if FORCE_RL_GUARDIAN_AS_BEST_MODEL and "V12B_RL_GUARDIAN_DAILY" in globals() and not pd.Series(V12B_RL_GUARDIAN_DAILY).empty:
    BEST_MODEL_NAME = "v12b_small_rl_guardian"
    BEST_MODEL_RETURNS = pd.Series(V12B_RL_GUARDIAN_DAILY).copy().rename(BEST_MODEL_NAME)
    BEST_MODEL_SUMMARY = V12B_RL_GUARDIAN_SUMMARY.copy() if "V12B_RL_GUARDIAN_SUMMARY" in globals() else pd.DataFrame()
    BEST_MODEL_SELECTED_DECISIONS = V12B_SELECTED_DECISIONS.copy() if "V12B_SELECTED_DECISIONS" in globals() else pd.DataFrame()
    print("Production lock active. BEST_MODEL:", BEST_MODEL_NAME)


In [ ]:
# Production preflight: make sure the active model is the RL Guardian version
if BEST_MODEL_NAME != "v12b_small_rl_guardian":
    raise RuntimeError(f"Wrong model selected: {BEST_MODEL_NAME}. Expected v12b_small_rl_guardian.")

if "BEST_MODEL_RETURNS" not in globals() or pd.Series(BEST_MODEL_RETURNS).empty:
    raise RuntimeError("BEST_MODEL_RETURNS is empty. Load or rebuild the RL Guardian daily returns first.")

if LIVE_TRADING_ENABLED:
    raise RuntimeError("LIVE_TRADING_ENABLED must stay False until paper-trading validation passes.")

checks = {
    "active_best_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "n_return_days": int(len(pd.Series(BEST_MODEL_RETURNS))),
    "has_selected_decisions": bool("BEST_MODEL_SELECTED_DECISIONS" in globals() and isinstance(BEST_MODEL_SELECTED_DECISIONS, pd.DataFrame) and not BEST_MODEL_SELECTED_DECISIONS.empty),
    "has_live_order_map": bool("V12B_LIVE_ORDER_MAP" in globals() and isinstance(V12B_LIVE_ORDER_MAP, pd.DataFrame) and not V12B_LIVE_ORDER_MAP.empty),
    "live_trading_enabled": LIVE_TRADING_ENABLED,
}

display(pd.Series(checks).to_frame("preflight"))
print("Preflight passed for artifact/backtest export.")


In [ ]:
# V12B SMALL RL GUARDIAN - FULL BACKTEST EXPORT IN REFERENCE FOLDER FORMAT
# Creates the same 8 CSV filenames/column formats as the Drive reference folder.
# Run this AFTER the V12B activation cell and the small RL Guardian cell.

import json, math, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

EXPORT_INITIAL_CAPITAL = float(globals().get("EXPORT_INITIAL_CAPITAL", 100000.0))
EXPORT_RISK_FREE_RATE = float(globals().get("EXPORT_RISK_FREE_RATE", 0.045))
EXPORT_ANNUALIZATION_DAYS = int(globals().get("EXPORT_ANNUALIZATION_DAYS", 252))

BEST_MODEL_NAME = "v12b_small_rl_guardian"
BEST_MODEL_VERSION = "v12b_exact_same_leg_inverse_plus_small_rl_guardian_v1"
STRATEGY_MODE = "RL Guardian Exact Same-Leg Inverse"
STRATEGY_NAME = "V12B Exact Same-Leg Inverse + Small RL Guardian"
ASSET_SYMBOL = "US_EQ_OPTIONS_EVENT_VOL_BASKET"
MARKET_COVERED = "US listed equity options earnings/event-vol basket using exact same-leg inverse execution and RL Guardian sizing."

if "V12B_ROOT" not in globals():
    if "V6_ROOT" in globals():
        V12B_ROOT = Path(V6_ROOT) / "v12b_exact_same_leg_inverse"
    else:
        V12B_ROOT = Path("/content/drive/MyDrive/quant_event_vol_world_model_v2/outputs_brppo_guardian/brppo_guardian_execution/v12b_exact_same_leg_inverse")

FINAL_BACKTEST_ROOT = V12B_ROOT / "final_full_backtest_results"
FINAL_BACKTEST_ROOT.mkdir(parents=True, exist_ok=True)

def _date_index(values):
    x = pd.to_datetime(values, errors="coerce", utc=True)
    if isinstance(x, pd.Series):
        return x.dt.tz_convert(None).dt.normalize()
    return x.tz_convert(None).normalize()

def _series_from_any(obj, preferred_name=None):
    if isinstance(obj, pd.Series):
        s = obj.copy()
    elif isinstance(obj, pd.DataFrame):
        cols = [preferred_name, "v12b_small_rl_guardian", "portfolio_ret", "daily_return", "return", "ret", "daily_ret"]
        col = next((c for c in cols if c and c in obj.columns), None)
        if col is None:
            num_cols = obj.select_dtypes(include=[np.number]).columns.tolist()
            if not num_cols:
                raise RuntimeError("Could not find numeric return column in model return object.")
            col = num_cols[0]
        date_col = next((c for c in ["date", "path_date", "timestamp_utc", "timestamp"] if c in obj.columns), None)
        idx = obj[date_col] if date_col else obj.index
        s = pd.Series(obj[col].values, index=idx, name=preferred_name or col)
    else:
        s = pd.Series(obj)

    s.index = _date_index(s.index)
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    s = s[~pd.isna(s.index)]
    return s.groupby(level=0).sum().sort_index()

if "V12B_RL_GUARDIAN_DAILY" in globals():
    best_daily = _series_from_any(V12B_RL_GUARDIAN_DAILY, BEST_MODEL_NAME)
elif "BEST_MODEL_RETURNS" in globals() and str(globals().get("BEST_MODEL_NAME", "")) == BEST_MODEL_NAME:
    best_daily = _series_from_any(BEST_MODEL_RETURNS, BEST_MODEL_NAME)
else:
    raise RuntimeError("RL Guardian daily returns not found. Run the small RL Guardian cell first.")

BEST_MODEL_RETURNS = best_daily.copy()
BEST_MODEL_RETURNS.name = BEST_MODEL_NAME

if BEST_MODEL_RETURNS.empty:
    raise RuntimeError("BEST_MODEL_RETURNS is empty.")

run_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
BACKTEST_RUN_ID = f"v12b_rl_guardian_backtest_{run_ts}"

def _fmt_ts(d):
    return pd.Timestamp(d).strftime("%Y-%m-%dT00:00:00Z")

def _fmt_pos_ts(d):
    return pd.Timestamp(d).strftime("%Y-%m-%d 00:00:00+00:00")

def _safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def _metrics_from_returns(r, name):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)
    if n == 0:
        return {}
    eq = EXPORT_INITIAL_CAPITAL * (1.0 + r).cumprod()
    dd = eq / eq.cummax() - 1.0
    years = n / EXPORT_ANNUALIZATION_DAYS
    ending = float(eq.iloc[-1])
    ann_ret = (ending / EXPORT_INITIAL_CAPITAL) ** (1.0 / years) - 1.0 if years > 0 and ending > 0 else np.nan
    ann_vol = float(r.std(ddof=0) * np.sqrt(EXPORT_ANNUALIZATION_DAYS))
    sharpe0 = float(r.mean() / r.std(ddof=0) * np.sqrt(EXPORT_ANNUALIZATION_DAYS)) if r.std(ddof=0) > 0 else np.nan
    sharpe_excess = (ann_ret - EXPORT_RISK_FREE_RATE) / ann_vol if ann_vol > 0 else np.nan
    downside = r[r < 0].std(ddof=0) * np.sqrt(EXPORT_ANNUALIZATION_DAYS)
    sortino = (ann_ret - EXPORT_RISK_FREE_RATE) / downside if downside and downside > 0 else np.nan
    max_dd = abs(float(dd.min()))
    calmar = ann_ret / max_dd if max_dd > 0 else np.nan
    var95 = float(np.quantile(r, 0.05))
    cvar95 = float(r[r <= var95].mean()) if (r <= var95).any() else np.nan

    return {
        "ending_portfolio_value": ending,
        "total_pnl": ending - EXPORT_INITIAL_CAPITAL,
        "total_return_pct": (ending / EXPORT_INITIAL_CAPITAL - 1.0) * 100.0,
        "annualized_return_pct": ann_ret * 100.0,
        "annualized_volatility_pct": ann_vol * 100.0,
        "daily_mean_return_pct": float(r.mean() * 100.0),
        "cumulative_return_pct": (ending / EXPORT_INITIAL_CAPITAL - 1.0) * 100.0,
        "max_drawdown_pct": max_dd * 100.0,
        "sharpe_ratio": sharpe0,
        "sharpe_manual_rf0": sharpe0,
        "sharpe_manual_excess_rf": sharpe_excess,
        "sortino_ratio": sortino,
        "sortino_manual_excess_rf": sortino,
        "calmar_ratio": calmar,
        "var_95_daily_return_pct": var95 * 100.0,
        "cvar_95_daily_return_pct": cvar95 * 100.0,
    }

def _drawdown_episodes(equity):
    rows = []
    peak_date = equity.index[0]
    peak_val = float(equity.iloc[0])
    in_dd = False
    start_date = valley_date = None
    valley_val = None

    for d, v in equity.items():
        v = float(v)
        if v >= peak_val:
            if in_dd:
                rows.append({
                    "Drawdown Id": len(rows),
                    "Column": 0,
                    "Peak Timestamp": _fmt_pos_ts(peak_date),
                    "Start Timestamp": _fmt_pos_ts(start_date),
                    "Valley Timestamp": _fmt_pos_ts(valley_date),
                    "End Timestamp": _fmt_pos_ts(d),
                    "Peak Value": peak_val,
                    "Valley Value": valley_val,
                    "End Value": v,
                    "Status": "Recovered",
                })
            peak_date, peak_val = d, v
            in_dd = False
        else:
            if not in_dd:
                start_date, valley_date, valley_val = d, d, v
                in_dd = True
            elif v < valley_val:
                valley_date, valley_val = d, v

    if in_dd:
        rows.append({
            "Drawdown Id": len(rows),
            "Column": 0,
            "Peak Timestamp": _fmt_pos_ts(peak_date),
            "Start Timestamp": _fmt_pos_ts(start_date),
            "Valley Timestamp": _fmt_pos_ts(valley_date),
            "End Timestamp": _fmt_pos_ts(equity.index[-1]),
            "Peak Value": peak_val,
            "Valley Value": valley_val,
            "End Value": float(equity.iloc[-1]),
            "Status": "Active",
        })
    return pd.DataFrame(rows)

def _guardian_multiplier():
    if "V12B_RL_GUARDIAN_ACTIONS" not in globals():
        return pd.Series(np.where(BEST_MODEL_RETURNS.abs() > 0, 1.0, 0.0), index=BEST_MODEL_RETURNS.index)

    g = pd.DataFrame(V12B_RL_GUARDIAN_ACTIONS).copy()
    if g.empty:
        return pd.Series(np.where(BEST_MODEL_RETURNS.abs() > 0, 1.0, 0.0), index=BEST_MODEL_RETURNS.index)

    date_col = next((c for c in ["date", "path_date", "timestamp_utc", "timestamp"] if c in g.columns), None)
    idx = _date_index(g[date_col]) if date_col else _date_index(g.index)
    mult_col = next((c for c in ["multiplier", "rl_multiplier", "guardian_multiplier", "chosen_multiplier", "target_position", "position_multiplier"] if c in g.columns), None)

    if mult_col is None:
        m = pd.Series(np.where(BEST_MODEL_RETURNS.abs() > 0, 1.0, 0.0), index=BEST_MODEL_RETURNS.index)
    else:
        m = pd.Series(pd.to_numeric(g[mult_col], errors="coerce").fillna(0.0).values, index=idx)
        m = m.groupby(level=0).mean().reindex(BEST_MODEL_RETURNS.index).fillna(0.0)
    return m.clip(lower=0.0)

guardian_mult = _guardian_multiplier()

cash_daily = pd.Series((1.0 + EXPORT_RISK_FREE_RATE) ** (1.0 / EXPORT_ANNUALIZATION_DAYS) - 1.0, index=BEST_MODEL_RETURNS.index)
equity = EXPORT_INITIAL_CAPITAL * (1.0 + BEST_MODEL_RETURNS).cumprod()
cash_equity = EXPORT_INITIAL_CAPITAL * (1.0 + cash_daily).cumprod()
drawdown = equity / equity.cummax() - 1.0
cash_drawdown = cash_equity / cash_equity.cummax() - 1.0

# Signal-level export
if "BEST_MODEL_SELECTED_DECISIONS" in globals():
    decisions = pd.DataFrame(BEST_MODEL_SELECTED_DECISIONS).copy()
elif "V12B_SELECTED_DECISIONS" in globals():
    decisions = pd.DataFrame(V12B_SELECTED_DECISIONS).copy()
else:
    decisions = pd.DataFrame()

if not decisions.empty:
    date_col = next((c for c in ["entry_date", "date", "timestamp_utc", "timestamp"] if c in decisions.columns), None)
    ticker_col = "ticker" if "ticker" in decisions.columns else None
    conf_col = next((c for c in ["confidence", "pred_win", "pred_mean", "score"] if c in decisions.columns), None)
    action_col = "action" if "action" in decisions.columns else None

    sig_dates = _date_index(decisions[date_col]) if date_col else _date_index(decisions.index)
    sig = []
    for _, row in decisions.iterrows():
        action = str(row.get(action_col, "")).lower() if action_col else ""
        if action == "long_vol":
            sig.append(-1)
        elif action == "short_vol_defined":
            sig.append(1)
        else:
            sig.append(0)

    signal_df = pd.DataFrame({
        "timestamp_utc": [_fmt_ts(d) for d in sig_dates],
        "backtest_run_id": BACKTEST_RUN_ID,
        "model_name": BEST_MODEL_NAME,
        "model_version": BEST_MODEL_VERSION,
        "asset_symbol": decisions[ticker_col].astype(str).values if ticker_col else ASSET_SYMBOL,
        "market_covered": MARKET_COVERED,
        "signal": sig,
        "confidence": pd.to_numeric(decisions[conf_col], errors="coerce").fillna(1.0).values if conf_col else 1.0,
    })
else:
    signal_df = pd.DataFrame({
        "timestamp_utc": [_fmt_ts(d) for d in BEST_MODEL_RETURNS.index],
        "backtest_run_id": BACKTEST_RUN_ID,
        "model_name": BEST_MODEL_NAME,
        "model_version": BEST_MODEL_VERSION,
        "asset_symbol": ASSET_SYMBOL,
        "market_covered": MARKET_COVERED,
        "signal": np.where(BEST_MODEL_RETURNS.abs() > 0, 1, 0),
        "confidence": guardian_mult.values,
    })

# Trade/order/position logs
def _load_paths():
    for nm in ["v12b_paths", "V12B_PATHS", "V12B_INVERTED_PATHS", "inv_paths"]:
        if nm in globals():
            df = pd.DataFrame(globals()[nm]).copy()
            if not df.empty:
                return df, False
    for p in [V12B_ROOT / "v12b_paths.parquet", V12B_ROOT / "v12b_exact_inverse_paths.parquet", V12B_ROOT / "v12b_exact_same_leg_inverse_paths.parquet"]:
        if p.exists():
            return pd.read_parquet(p), False
    if "v6_real_paths" in globals():
        df = pd.DataFrame(v6_real_paths).copy()
        if not df.empty:
            return df, True
    return pd.DataFrame(), False

paths, source_is_original = _load_paths()
trade_rows, order_rows, position_rows = [], [], []

if not paths.empty and {"path_date", "entry_date"}.issubset(paths.columns):
    paths = paths.copy()
    paths["path_date"] = _date_index(paths["path_date"])
    paths["entry_date"] = _date_index(paths["entry_date"])
    paths = paths[paths["path_date"].isin(BEST_MODEL_RETURNS.index)].copy()

    if not paths.empty:
        active_counts = paths.groupby("path_date").size().replace(0, np.nan)
        paths["_alloc_daily_return"] = paths["path_date"].map(BEST_MODEL_RETURNS) / paths["path_date"].map(active_counts)
        paths["_alloc_daily_pnl"] = paths["_alloc_daily_return"].fillna(0.0) * EXPORT_INITIAL_CAPITAL

        group_cols = [c for c in ["ticker", "entry_date", "action", "short_call", "short_put", "long_call_wing", "long_put_wing"] if c in paths.columns]
        if not group_cols:
            paths["_trade_id"] = np.arange(len(paths))
            group_cols = ["_trade_id"]

        for i, (_, g) in enumerate(paths.groupby(group_cols, dropna=False)):
            g = g.sort_values("path_date")
            ticker = str(g["ticker"].iloc[0]) if "ticker" in g.columns else ASSET_SYMBOL
            action = str(g["action"].iloc[0]).lower() if "action" in g.columns else ""
            direction = "Short" if action == "long_vol" else "Long" if action == "short_vol_defined" else "Long"
            entry_ts = g["path_date"].min()
            exit_ts = g["path_date"].max()
            pnl = float(g["_alloc_daily_pnl"].sum())
            ret_trade = pnl / EXPORT_INITIAL_CAPITAL

            entry_price = np.nan
            for c in ["entry_debit", "entry_credit", "spot_entry", "entry_price"]:
                if c in g.columns:
                    entry_price = _safe_float(g[c].dropna().iloc[0]) if not g[c].dropna().empty else np.nan
                    if not pd.isna(entry_price):
                        break
            exit_price = entry_price

            size = np.nan
            for c in ["trade_weight", "margin_required_pct", "max_loss_u"]:
                if c in g.columns:
                    size = abs(_safe_float(g[c].dropna().iloc[0])) if not g[c].dropna().empty else np.nan
                    if not pd.isna(size):
                        break
            if pd.isna(size) or size == 0:
                size = 1.0

            base = {
                "backtest_run_id": BACKTEST_RUN_ID,
                "period_name": "Full Available",
                "test_type": "final_best_model",
                "model_name": BEST_MODEL_NAME,
                "model_version": BEST_MODEL_VERSION,
                "strategy_mode": STRATEGY_MODE,
                "strategy_name": STRATEGY_NAME,
                "asset_symbol": ticker,
                "market_covered": MARKET_COVERED,
            }

            trade_rows.append({
                **base,
                "Exit Trade Id": i,
                "Column": 0,
                "Size": size,
                "Entry Timestamp": _fmt_pos_ts(entry_ts),
                "Avg Entry Price": entry_price,
                "Entry Fees": 0.0,
                "Exit Timestamp": _fmt_pos_ts(exit_ts),
                "Avg Exit Price": exit_price,
                "Exit Fees": 0.0,
                "PnL": pnl,
                "Return": ret_trade,
                "Direction": direction,
                "Status": "Closed",
                "Position Id": i,
                "Return %": ret_trade * 100.0,
            })

            position_rows.append({
                **base,
                "Position Id": i,
                "Column": 0,
                "Size": size,
                "Entry Timestamp": _fmt_pos_ts(entry_ts),
                "Avg Entry Price": entry_price,
                "Entry Fees": 0.0,
                "Exit Timestamp": _fmt_pos_ts(exit_ts),
                "Avg Exit Price": exit_price,
                "Exit Fees": 0.0,
                "PnL": pnl,
                "Return": ret_trade,
                "Direction": direction,
                "Status": "Closed",
            })

            entry_side = "Sell" if direction == "Short" else "Buy"
            exit_side = "Buy" if direction == "Short" else "Sell"
            order_rows.append({
                **base,
                "Order Id": len(order_rows),
                "Column": 0,
                "Timestamp": _fmt_pos_ts(entry_ts),
                "Size": size,
                "Price": entry_price,
                "Fees": 0.0,
                "Side": entry_side,
            })
            order_rows.append({
                **base,
                "Order Id": len(order_rows),
                "Column": 0,
                "Timestamp": _fmt_pos_ts(exit_ts),
                "Size": size,
                "Price": exit_price,
                "Fees": 0.0,
                "Side": exit_side,
            })

trade_cols = ["backtest_run_id","period_name","test_type","model_name","model_version","strategy_mode","strategy_name","asset_symbol","market_covered","Exit Trade Id","Column","Size","Entry Timestamp","Avg Entry Price","Entry Fees","Exit Timestamp","Avg Exit Price","Exit Fees","PnL","Return","Direction","Status","Position Id","Return %"]
order_cols = ["backtest_run_id","period_name","test_type","model_name","model_version","strategy_mode","strategy_name","asset_symbol","market_covered","Order Id","Column","Timestamp","Size","Price","Fees","Side"]
position_cols = ["backtest_run_id","period_name","test_type","model_name","model_version","strategy_mode","strategy_name","asset_symbol","market_covered","Position Id","Column","Size","Entry Timestamp","Avg Entry Price","Entry Fees","Exit Timestamp","Avg Exit Price","Exit Fees","PnL","Return","Direction","Status"]

trade_log = pd.DataFrame(trade_rows, columns=trade_cols)
order_log = pd.DataFrame(order_rows, columns=order_cols)
position_log = pd.DataFrame(position_rows, columns=position_cols)

orders_by_date = pd.Series(dtype=float)
closed_by_date = pd.Series(dtype=float)
pnl_by_date = pd.Series(dtype=float)

if not order_log.empty:
    od = _date_index(order_log["Timestamp"])
    orders_by_date = pd.Series(1, index=od).groupby(level=0).sum()

if not trade_log.empty:
    td = _date_index(trade_log["Exit Timestamp"])
    closed_by_date = pd.Series(1, index=td).groupby(level=0).sum()
    pnl_by_date = pd.Series(pd.to_numeric(trade_log["PnL"], errors="coerce").fillna(0.0).values, index=td).groupby(level=0).sum()

daily_pnl = equity.diff().fillna(equity.iloc[0] - EXPORT_INITIAL_CAPITAL)
rolling_30d_sharpe = BEST_MODEL_RETURNS.rolling(30).mean() / BEST_MODEL_RETURNS.rolling(30).std(ddof=0) * np.sqrt(EXPORT_ANNUALIZATION_DAYS)

daily_log = pd.DataFrame({
    "backtest_run_id": BACKTEST_RUN_ID,
    "timestamp_utc": [_fmt_ts(d) for d in BEST_MODEL_RETURNS.index],
    "date": [pd.Timestamp(d).strftime("%Y-%m-%d") for d in BEST_MODEL_RETURNS.index],
    "period_name": "Full Available",
    "test_type": "final_best_model",
    "model_name": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "strategy_mode": STRATEGY_MODE,
    "strategy_name": STRATEGY_NAME,
    "asset_symbol": ASSET_SYMBOL,
    "market_covered": MARKET_COVERED,
    "data_frequency": "D",
    "trading_frequency": "Event-driven options entries with daily path marking and RL Guardian sizing",
    "open": np.nan,
    "high": np.nan,
    "low": np.nan,
    "close": np.nan,
    "volume": np.nan,
    "signal": np.where(BEST_MODEL_RETURNS.abs() > 0, 1.0, 0.0),
    "confidence": guardian_mult.reindex(BEST_MODEL_RETURNS.index).fillna(0.0).values,
    "target_position": guardian_mult.reindex(BEST_MODEL_RETURNS.index).fillna(0.0).values,
    "portfolio_value": equity.values,
    "cash": np.nan,
    "asset_units": np.nan,
    "asset_value": np.nan,
    "signed_exposure_pct": guardian_mult.reindex(BEST_MODEL_RETURNS.index).fillna(0.0).values * 100.0,
    "gross_exposure_pct": guardian_mult.reindex(BEST_MODEL_RETURNS.index).fillna(0.0).abs().values * 100.0,
    "daily_return": BEST_MODEL_RETURNS.values,
    "cumulative_return": (equity / EXPORT_INITIAL_CAPITAL - 1.0).values,
    "daily_pnl": daily_pnl.values,
    "cumulative_pnl": (equity - EXPORT_INITIAL_CAPITAL).values,
    "drawdown_pct": abs(drawdown.values) * 100.0,
    "rolling_30d_sharpe": rolling_30d_sharpe.values,
    "fee_rate": 0.0,
    "slippage_rate": 0.0,
    "total_cost_rate": 0.0,
    "fees_paid": 0.0,
    "orders_count": [orders_by_date.get(d, 0) for d in BEST_MODEL_RETURNS.index],
    "closed_trades": [closed_by_date.get(d, 0) for d in BEST_MODEL_RETURNS.index],
    "realized_trade_pnl": [pnl_by_date.get(d, 0.0) for d in BEST_MODEL_RETURNS.index],
    "benchmark_portfolio_value": cash_equity.values,
    "benchmark_daily_return": cash_daily.values,
    "benchmark_cumulative_return": (cash_equity / EXPORT_INITIAL_CAPITAL - 1.0).values,
    "benchmark_drawdown_pct": abs(cash_drawdown.values) * 100.0,
})

drawdown_log = _drawdown_episodes(equity)
drawdown_log.insert(0, "market_covered", MARKET_COVERED)
drawdown_log.insert(0, "asset_symbol", ASSET_SYMBOL)
drawdown_log.insert(0, "strategy_name", STRATEGY_NAME)
drawdown_log.insert(0, "strategy_mode", STRATEGY_MODE)
drawdown_log.insert(0, "model_version", BEST_MODEL_VERSION)
drawdown_log.insert(0, "model_name", BEST_MODEL_NAME)
drawdown_log.insert(0, "test_type", "final_best_model")
drawdown_log.insert(0, "period_name", "Full Available")
drawdown_log.insert(0, "backtest_run_id", BACKTEST_RUN_ID)

m = _metrics_from_returns(BEST_MODEL_RETURNS, BEST_MODEL_NAME)
bm = _metrics_from_returns(cash_daily, "cash")

trade_pnl = pd.to_numeric(trade_log["PnL"], errors="coerce").dropna() if not trade_log.empty else pd.Series(dtype=float)
gross_profit = float(trade_pnl[trade_pnl > 0].sum()) if not trade_pnl.empty else float(daily_pnl[daily_pnl > 0].sum())
gross_loss = float(abs(trade_pnl[trade_pnl < 0].sum())) if not trade_pnl.empty else float(abs(daily_pnl[daily_pnl < 0].sum()))
profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.nan
win_rate = float((trade_pnl > 0).mean() * 100.0) if len(trade_pnl) else np.nan

metrics_cols = ["backtest_run_id","period_name","test_type","requested_start","requested_end","backtest_start","backtest_end","data_frequency","trading_frequency","asset_symbol","market_covered","model_name","model_version","strategy_mode","strategy_name","initial_capital","ending_portfolio_value","total_pnl","total_return_pct","annualized_return_pct","annualized_volatility_pct","daily_mean_return_pct","cumulative_return_pct","max_drawdown_pct","sharpe_ratio","sharpe_manual_rf0","sharpe_manual_excess_rf","sortino_ratio","sortino_manual_excess_rf","calmar_ratio","var_95_daily_return_pct","cvar_95_daily_return_pct","number_of_orders","number_of_trades","closed_trades","win_rate_pct","average_profit_loss_per_trade","average_winning_trade_pnl","average_losing_trade_pnl","gross_profit","gross_loss","profit_factor","total_fees_paid","fee_rate","slippage_rate","total_cost_rate","leverage","position_sizing","risk_limits","annualization_days","risk_free_rate_annual","benchmark_strategy","benchmark_total_return_pct","benchmark_annualized_return_pct","benchmark_annualized_volatility_pct","benchmark_sharpe_ratio","benchmark_sortino_ratio","benchmark_max_drawdown_pct"]

metrics_df = pd.DataFrame([{
    "backtest_run_id": BACKTEST_RUN_ID,
    "period_name": "Full Available",
    "test_type": "final_best_model",
    "requested_start": BEST_MODEL_RETURNS.index.min().strftime("%Y-%m-%d"),
    "requested_end": BEST_MODEL_RETURNS.index.max().strftime("%Y-%m-%d"),
    "backtest_start": BEST_MODEL_RETURNS.index.min().strftime("%Y-%m-%d"),
    "backtest_end": BEST_MODEL_RETURNS.index.max().strftime("%Y-%m-%d"),
    "data_frequency": "D",
    "trading_frequency": "Event-driven options entries with daily path marking and RL Guardian sizing",
    "asset_symbol": ASSET_SYMBOL,
    "market_covered": MARKET_COVERED,
    "model_name": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "strategy_mode": STRATEGY_MODE,
    "strategy_name": STRATEGY_NAME,
    "initial_capital": EXPORT_INITIAL_CAPITAL,
    **m,
    "number_of_orders": int(len(order_log)),
    "number_of_trades": int(len(trade_log)),
    "closed_trades": int((trade_log["Status"] == "Closed").sum()) if not trade_log.empty else 0,
    "win_rate_pct": win_rate,
    "average_profit_loss_per_trade": float(trade_pnl.mean()) if len(trade_pnl) else np.nan,
    "average_winning_trade_pnl": float(trade_pnl[trade_pnl > 0].mean()) if (trade_pnl > 0).any() else np.nan,
    "average_losing_trade_pnl": float(trade_pnl[trade_pnl < 0].mean()) if (trade_pnl < 0).any() else np.nan,
    "gross_profit": gross_profit,
    "gross_loss": gross_loss,
    "profit_factor": profit_factor,
    "total_fees_paid": 0.0,
    "fee_rate": 0.0,
    "slippage_rate": 0.0,
    "total_cost_rate": 0.0,
    "leverage": "Options spread/straddle exposure controlled by RL Guardian multiplier; see live artifact for leg-level mapping.",
    "position_sizing": "V12B exact same-leg inverse returns scaled by small RL Guardian state-action sizing overlay.",
    "risk_limits": "RL Guardian drawdown/volatility/margin state controls. Live trading remains disabled until broker order validation.",
    "annualization_days": EXPORT_ANNUALIZATION_DAYS,
    "risk_free_rate_annual": EXPORT_RISK_FREE_RATE,
    "benchmark_strategy": "cash_4.5pct_same_period",
    "benchmark_total_return_pct": bm.get("total_return_pct", np.nan),
    "benchmark_annualized_return_pct": bm.get("annualized_return_pct", np.nan),
    "benchmark_annualized_volatility_pct": bm.get("annualized_volatility_pct", np.nan),
    "benchmark_sharpe_ratio": bm.get("sharpe_ratio", np.nan),
    "benchmark_sortino_ratio": bm.get("sortino_ratio", np.nan),
    "benchmark_max_drawdown_pct": bm.get("max_drawdown_pct", np.nan),
}], columns=metrics_cols)

errors_df = pd.DataFrame(columns=["timestamp_utc", "severity", "source", "message"])

exports = {
    "all_periods_all_models_metrics.csv": metrics_df,
    "all_periods_all_models_daily_backtest.csv": daily_log,
    "all_models_signal_level.csv": signal_df,
    "all_periods_all_models_trade_log.csv": trade_log,
    "all_periods_all_models_order_log.csv": order_log,
    "all_periods_all_models_position_log.csv": position_log,
    "all_periods_all_models_drawdown_log.csv": drawdown_log,
    "backtest_export_errors.csv": errors_df,
}

for name, df in exports.items():
    df.to_csv(FINAL_BACKTEST_ROOT / name, index=False)

manifest = {
    "backtest_run_id": BACKTEST_RUN_ID,
    "selected_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "export_format_source": "Google Drive reference folder 11l3yabXKWod9WvDpuE8StTTt3IB3IEpX",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "files": {k: {"rows": int(len(v)), "columns": list(v.columns)} for k, v in exports.items()},
}
with open(FINAL_BACKTEST_ROOT / "backtest_export_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

FINAL_BACKTEST_ZIP = V12B_ROOT / "v12b_small_rl_guardian_reference_format_backtest_export.zip"
with zipfile.ZipFile(FINAL_BACKTEST_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in exports:
        z.write(FINAL_BACKTEST_ROOT / name, arcname=name)
    z.write(FINAL_BACKTEST_ROOT / "backtest_export_manifest.json", arcname="backtest_export_manifest.json")

display(metrics_df)
display(pd.DataFrame([{"file": k, "rows": len(v), "cols": len(v.columns)} for k, v in exports.items()]))

print("Saved reference-format backtest export to:", FINAL_BACKTEST_ROOT)
print("Saved zip:", FINAL_BACKTEST_ZIP)
print("Active BEST_MODEL:", BEST_MODEL_NAME)


In [ ]:
# FINAL MODEL ARTIFACT EXPORT - FORCE V12B SMALL RL GUARDIAN
# Run AFTER:
# 1. V12B activation cell
# 2. Small RL Guardian cell
# 3. Reference-format full backtest export cell

import json, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

BEST_MODEL_NAME = "v12b_small_rl_guardian"
BEST_MODEL_VERSION = "v12b_exact_same_leg_inverse_plus_small_rl_guardian_v1"
BEST_STRATEGY_NAME = "V12B Exact Same-Leg Inverse + Small RL Guardian"
BEST_STRATEGY_MODE = "RL Guardian Exact Same-Leg Inverse"
BEST_ASSET_SYMBOL = "US_EQ_OPTIONS_EVENT_VOL_BASKET"
BEST_MARKET_COVERED = "US listed equity options earnings/event-vol basket using exact same-leg inverse execution and RL Guardian sizing."
LIVE_TRADING_ENABLED = False

if "V12B_ROOT" not in globals():
    if "V6_ROOT" in globals():
        V12B_ROOT = Path(V6_ROOT) / "v12b_exact_same_leg_inverse"
    else:
        V12B_ROOT = Path("/content/drive/MyDrive/quant_event_vol_world_model_v2/outputs_brppo_guardian/brppo_guardian_execution/v12b_exact_same_leg_inverse")

FINAL_ARTIFACT_ROOT = V12B_ROOT / "final_full_model_artifact"
FINAL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def _date_index(values):
    x = pd.to_datetime(values, errors="coerce", utc=True)
    if isinstance(x, pd.Series):
        return x.dt.tz_convert(None).dt.normalize()
    return x.tz_convert(None).normalize()

def _series_from_any(obj, preferred_name):
    if isinstance(obj, pd.Series):
        s = obj.copy()
    elif isinstance(obj, pd.DataFrame):
        col = next((c for c in [preferred_name, "v12b_small_rl_guardian", "portfolio_ret", "daily_return", "return", "ret"] if c in obj.columns), None)
        if col is None:
            num_cols = obj.select_dtypes(include=[np.number]).columns.tolist()
            if not num_cols:
                raise RuntimeError("No numeric return column found for RL Guardian daily returns.")
            col = num_cols[0]
        date_col = next((c for c in ["date", "path_date", "timestamp_utc", "timestamp"] if c in obj.columns), None)
        idx = obj[date_col] if date_col else obj.index
        s = pd.Series(obj[col].values, index=idx, name=preferred_name)
    else:
        s = pd.Series(obj, name=preferred_name)

    s.index = _date_index(s.index)
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    s = s[~pd.isna(s.index)]
    return s.groupby(level=0).sum().sort_index().rename(preferred_name)

if "V12B_RL_GUARDIAN_DAILY" in globals():
    BEST_MODEL_RETURNS = _series_from_any(V12B_RL_GUARDIAN_DAILY, BEST_MODEL_NAME)
elif "BEST_MODEL_RETURNS" in globals() and str(globals().get("BEST_MODEL_NAME", "")) == BEST_MODEL_NAME:
    BEST_MODEL_RETURNS = _series_from_any(BEST_MODEL_RETURNS, BEST_MODEL_NAME)
else:
    candidate_paths = [
        V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_daily_returns.csv",
        V12B_ROOT / "small_rl_guardian" / "v12b_small_rl_guardian_daily.csv",
        V12B_ROOT / "v12b_small_rl_guardian_daily_returns.csv",
    ]
    found = next((p for p in candidate_paths if p.exists()), None)
    if found is None:
        raise RuntimeError("RL Guardian daily returns not found. Run the small RL Guardian cell first.")
    raw = pd.read_csv(found)
    BEST_MODEL_RETURNS = _series_from_any(raw, BEST_MODEL_NAME)

if BEST_MODEL_RETURNS.empty:
    raise RuntimeError("RL Guardian BEST_MODEL_RETURNS is empty.")

BEST_MODEL_NAME = "v12b_small_rl_guardian"

def _metrics(r):
    r = pd.Series(r).dropna().astype(float)
    eq = (1.0 + r).cumprod()
    dd = eq / eq.cummax() - 1.0
    n = len(r)
    years = n / 252.0
    final_eq = float(eq.iloc[-1])
    ann_ret = final_eq ** (1.0 / years) - 1.0 if years > 0 and final_eq > 0 else np.nan
    ann_vol = float(r.std(ddof=0) * np.sqrt(252))
    sharpe = float(r.mean() / r.std(ddof=0) * np.sqrt(252)) if r.std(ddof=0) > 0 else np.nan
    max_dd = abs(float(dd.min()))
    return {
        "annual_return": ann_ret,
        "annual_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "worst_day": float(r.min()),
        "best_day": float(r.max()),
        "final_equity": final_eq,
        "n_days": int(n),
    }

best_metrics = _metrics(BEST_MODEL_RETURNS)

# Pull final backtest metrics if the prior export cell ran.
backtest_metrics_path = globals().get("FINAL_BACKTEST_ROOT", V12B_ROOT / "final_full_backtest_results")
backtest_metrics_path = Path(backtest_metrics_path) / "all_periods_all_models_metrics.csv"
if backtest_metrics_path.exists():
    backtest_metrics_df = pd.read_csv(backtest_metrics_path)
else:
    backtest_metrics_df = pd.DataFrame([{
        "model_name": BEST_MODEL_NAME,
        "model_version": BEST_MODEL_VERSION,
        **best_metrics,
    }])

# Selected decisions
if "BEST_MODEL_SELECTED_DECISIONS" in globals():
    selected_decisions = pd.DataFrame(BEST_MODEL_SELECTED_DECISIONS).copy()
elif "V12B_SELECTED_DECISIONS" in globals():
    selected_decisions = pd.DataFrame(V12B_SELECTED_DECISIONS).copy()
elif "enriched" in globals():
    selected_decisions = pd.DataFrame(enriched).copy()
else:
    selected_decisions = pd.DataFrame()

if not selected_decisions.empty:
    if "entry_date" in selected_decisions.columns:
        selected_decisions["entry_date"] = pd.to_datetime(selected_decisions["entry_date"], errors="coerce").dt.normalize()
    selected_decisions["selected_model"] = BEST_MODEL_NAME
    selected_decisions["selected_model_version"] = BEST_MODEL_VERSION
    if "action" in selected_decisions.columns:
        selected_decisions["live_execution_action"] = selected_decisions["action"].map({
            "short_vol_defined": "take_opposite_side_of_same_iron_fly",
            "long_vol": "take_opposite_side_of_same_straddle",
        }).fillna("do_not_trade_unknown_action")

# Live order map
if "V12B_LIVE_ORDER_MAP" in globals():
    live_order_map = pd.DataFrame(V12B_LIVE_ORDER_MAP).copy()
else:
    live_order_map = pd.DataFrame()

if live_order_map.empty and "v6_real_paths" in globals():
    p = pd.DataFrame(v6_real_paths).copy()
    if not p.empty:
        key_cols = [c for c in ["ticker", "entry_date", "action", "short_call", "short_put", "long_call_wing", "long_put_wing"] if c in p.columns]
        if "entry_date" in p.columns:
            p["entry_date"] = pd.to_datetime(p["entry_date"], errors="coerce").dt.normalize()
        rows = []
        for _, g in p.drop_duplicates(key_cols).iterrows():
            action = str(g.get("action", ""))
            base = {
                "ticker": g.get("ticker"),
                "entry_date": g.get("entry_date"),
                "original_action": action,
                "selected_model": BEST_MODEL_NAME,
                "selected_model_version": BEST_MODEL_VERSION,
            }
            if action == "short_vol_defined":
                legs = [
                    ("BUY_TO_OPEN", g.get("short_call"), "original_short_call"),
                    ("BUY_TO_OPEN", g.get("short_put"), "original_short_put"),
                    ("SELL_TO_OPEN", g.get("long_call_wing"), "original_long_call_wing"),
                    ("SELL_TO_OPEN", g.get("long_put_wing"), "original_long_put_wing"),
                ]
                live_action = "take_opposite_side_of_same_iron_fly"
            elif action == "long_vol":
                legs = [
                    ("SELL_TO_OPEN", g.get("short_call"), "original_long_call_leg"),
                    ("SELL_TO_OPEN", g.get("short_put"), "original_long_put_leg"),
                ]
                live_action = "take_opposite_side_of_same_straddle"
            else:
                legs = []
                live_action = "do_not_trade_unknown_action"

            for leg_i, (side, contract, role) in enumerate(legs):
                if pd.notna(contract) and str(contract) not in ["None", "nan", ""]:
                    rows.append({
                        **base,
                        "live_execution_action": live_action,
                        "leg_number": leg_i + 1,
                        "order_side": side,
                        "option_ticker": contract,
                        "leg_role": role,
                    })
        live_order_map = pd.DataFrame(rows)

# Guardian outputs
guardian_actions = pd.DataFrame(V12B_RL_GUARDIAN_ACTIONS).copy() if "V12B_RL_GUARDIAN_ACTIONS" in globals() else pd.DataFrame()
guardian_summary = pd.DataFrame(V12B_RL_GUARDIAN_SUMMARY).copy() if "V12B_RL_GUARDIAN_SUMMARY" in globals() else pd.DataFrame()

# Save core artifact files
daily_df = BEST_MODEL_RETURNS.rename("daily_return").to_frame()
daily_df.insert(0, "date", daily_df.index.strftime("%Y-%m-%d"))
daily_df.insert(1, "model_name", BEST_MODEL_NAME)
daily_df.insert(2, "model_version", BEST_MODEL_VERSION)
daily_df["equity"] = (1.0 + BEST_MODEL_RETURNS).cumprod().values
daily_df["drawdown"] = daily_df["equity"] / daily_df["equity"].cummax() - 1.0

daily_df.to_csv(FINAL_ARTIFACT_ROOT / "model_daily_returns.csv", index=False)
backtest_metrics_df.to_csv(FINAL_ARTIFACT_ROOT / "backtest_metrics_used_for_artifact.csv", index=False)
selected_decisions.to_csv(FINAL_ARTIFACT_ROOT / "selected_decisions_live.csv", index=False)
live_order_map.to_csv(FINAL_ARTIFACT_ROOT / "live_order_map.csv", index=False)
guardian_actions.to_csv(FINAL_ARTIFACT_ROOT / "small_rl_guardian_actions.csv", index=False)
guardian_summary.to_csv(FINAL_ARTIFACT_ROOT / "small_rl_guardian_summary.csv", index=False)

deployment_action_mapping = {
    "selected_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "mandatory_live_mapping": {
        "short_vol_defined": {
            "live_action": "take_opposite_side_of_same_iron_fly",
            "legs": [
                "BUY_TO_OPEN original short call",
                "BUY_TO_OPEN original short put",
                "SELL_TO_OPEN original long call wing",
                "SELL_TO_OPEN original long put wing",
            ],
        },
        "long_vol": {
            "live_action": "take_opposite_side_of_same_straddle",
            "legs": [
                "SELL_TO_OPEN original call leg",
                "SELL_TO_OPEN original put leg",
            ],
        },
    },
    "live_trading_enabled": LIVE_TRADING_ENABLED,
    "deployment_blocker": "Broker order construction, margin, and live paper-trade validation required before enabling live trading.",
}

with open(FINAL_ARTIFACT_ROOT / "deployment_action_mapping.json", "w") as f:
    json.dump(deployment_action_mapping, f, indent=2, default=str)

live_state = {
    "selected_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "last_backtest_date": str(BEST_MODEL_RETURNS.index.max().date()),
    "live_trading_enabled": LIVE_TRADING_ENABLED,
    "current_equity_multiple": float((1.0 + BEST_MODEL_RETURNS).cumprod().iloc[-1]),
    "current_drawdown": float(((1.0 + BEST_MODEL_RETURNS).cumprod() / (1.0 + BEST_MODEL_RETURNS).cumprod().cummax() - 1.0).iloc[-1]),
    "risk_guardian": "small_rl_guardian",
    "active_return_source": "V12B_RL_GUARDIAN_DAILY",
    "order_mapping_required": "exact_same_leg_inverse",
}

with open(FINAL_ARTIFACT_ROOT / "live_state.json", "w") as f:
    json.dump(live_state, f, indent=2, default=str)

metadata = {
    "artifact_name": "V12B Small RL Guardian Model Artifact",
    "selected_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "strategy_name": BEST_STRATEGY_NAME,
    "strategy_mode": BEST_STRATEGY_MODE,
    "asset_symbol": BEST_ASSET_SYMBOL,
    "market_covered": BEST_MARKET_COVERED,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": best_metrics,
    "backtest_metrics_file": str(backtest_metrics_path),
    "live_trading_enabled": LIVE_TRADING_ENABLED,
    "important_note": "This artifact intentionally saves the RL Guardian model, not raw V12B.",
}

with open(FINAL_ARTIFACT_ROOT / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

model_card = f"""# {BEST_STRATEGY_NAME}

Selected model: `{BEST_MODEL_NAME}`
Model version: `{BEST_MODEL_VERSION}`
Live trading enabled: `{LIVE_TRADING_ENABLED}`

## Core Result

Annual return: `{best_metrics["annual_return"]:.6f}`
Annual volatility: `{best_metrics["annual_vol"]:.6f}`
Sharpe: `{best_metrics["sharpe"]:.6f}`
Max drawdown: `{best_metrics["max_drawdown"]:.6f}`
Worst day: `{best_metrics["worst_day"]:.6f}`
Final equity multiple: `{best_metrics["final_equity"]:.6f}`
Backtest days: `{best_metrics["n_days"]}`

## Live Mapping

`short_vol_defined` means take the opposite side of the exact same iron fly.
`long_vol` means take the opposite side of the exact same straddle.

This artifact is for research and paper-trading handoff until broker order construction, margin checks, and live validation are complete.
"""

with open(FINAL_ARTIFACT_ROOT / "MODEL_CARD.md", "w") as f:
    f.write(model_card)

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

artifact_files = sorted([p for p in FINAL_ARTIFACT_ROOT.iterdir() if p.is_file()])
manifest = {
    "selected_model": BEST_MODEL_NAME,
    "model_version": BEST_MODEL_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "artifact_root": str(FINAL_ARTIFACT_ROOT),
    "files": [
        {
            "name": p.name,
            "size_bytes": int(p.stat().st_size),
            "sha256": _sha256(p),
        }
        for p in artifact_files
    ],
}

with open(FINAL_ARTIFACT_ROOT / "artifact_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

FINAL_ARTIFACT_ZIP = V12B_ROOT / "v12b_small_rl_guardian_full_model_artifact.zip"
with zipfile.ZipFile(FINAL_ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FINAL_ARTIFACT_ROOT.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

display(pd.DataFrame([{
    "selected_model": BEST_MODEL_NAME,
    "annual_return": best_metrics["annual_return"],
    "annual_vol": best_metrics["annual_vol"],
    "sharpe": best_metrics["sharpe"],
    "max_drawdown": best_metrics["max_drawdown"],
    "worst_day": best_metrics["worst_day"],
    "final_equity": best_metrics["final_equity"],
    "n_days": best_metrics["n_days"],
}]))

display(pd.DataFrame([
    {"file": p.name, "size_bytes": int(p.stat().st_size)}
    for p in sorted(FINAL_ARTIFACT_ROOT.iterdir())
    if p.is_file()
]))

print("Saved RL Guardian model artifact to:", FINAL_ARTIFACT_ROOT)
print("Saved artifact zip:", FINAL_ARTIFACT_ZIP)
print("Active BEST_MODEL:", BEST_MODEL_NAME)


In [ ]:
# FINAL OUTPUT SUMMARY - V12B SMALL RL GUARDIAN ONLY

import json
from pathlib import Path
import numpy as np
import pandas as pd

BEST_MODEL_NAME = "v12b_small_rl_guardian"
BEST_MODEL_VERSION = "v12b_exact_same_leg_inverse_plus_small_rl_guardian_v1"

if "V12B_ROOT" not in globals():
    if "V6_ROOT" in globals():
        V12B_ROOT = Path(V6_ROOT) / "v12b_exact_same_leg_inverse"
    else:
        V12B_ROOT = Path("/content/drive/MyDrive/quant_event_vol_world_model_v2/outputs_brppo_guardian/brppo_guardian_execution/v12b_exact_same_leg_inverse")

FINAL_BACKTEST_ROOT = globals().get("FINAL_BACKTEST_ROOT", V12B_ROOT / "final_full_backtest_results")
FINAL_ARTIFACT_ROOT = globals().get("FINAL_ARTIFACT_ROOT", V12B_ROOT / "final_full_model_artifact")

metrics_path = Path(FINAL_BACKTEST_ROOT) / "all_periods_all_models_metrics.csv"
daily_path = Path(FINAL_BACKTEST_ROOT) / "all_periods_all_models_daily_backtest.csv"
artifact_meta_path = Path(FINAL_ARTIFACT_ROOT) / "model_metadata.json"
live_state_path = Path(FINAL_ARTIFACT_ROOT) / "live_state.json"

def _metric_from_daily(daily_returns):
    r = pd.Series(daily_returns).dropna().astype(float)
    eq = (1.0 + r).cumprod()
    dd = eq / eq.cummax() - 1.0
    n = len(r)
    years = n / 252.0
    final_eq = float(eq.iloc[-1])
    ann_ret = final_eq ** (1.0 / years) - 1.0 if years > 0 and final_eq > 0 else np.nan
    ann_vol = float(r.std(ddof=0) * np.sqrt(252))
    sharpe = float(r.mean() / r.std(ddof=0) * np.sqrt(252)) if r.std(ddof=0) > 0 else np.nan
    downside = r[r < 0].std(ddof=0) * np.sqrt(252)
    sortino = (ann_ret - 0.045) / downside if downside and downside > 0 else np.nan
    max_dd = abs(float(dd.min()))
    return {
        "model_name": BEST_MODEL_NAME,
        "model_version": BEST_MODEL_VERSION,
        "annual_return": ann_ret,
        "annual_return_pct": ann_ret * 100.0,
        "annual_vol": ann_vol,
        "annual_vol_pct": ann_vol * 100.0,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "max_drawdown_pct": max_dd * 100.0,
        "calmar": ann_ret / max_dd if max_dd > 0 else np.nan,
        "worst_day": float(r.min()),
        "worst_day_pct": float(r.min() * 100.0),
        "best_day": float(r.max()),
        "best_day_pct": float(r.max() * 100.0),
        "hit_rate": float((r > 0).mean()),
        "hit_rate_pct": float((r > 0).mean() * 100.0),
        "final_equity_multiple": final_eq,
        "n_days": int(n),
        "start_date": str(r.index.min().date()) if hasattr(r.index.min(), "date") else str(r.index.min()),
        "end_date": str(r.index.max().date()) if hasattr(r.index.max(), "date") else str(r.index.max()),
    }

if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    row = metrics_df.loc[metrics_df["model_name"].astype(str).eq(BEST_MODEL_NAME)].tail(1)
    if row.empty:
        row = metrics_df.tail(1)

    summary = pd.DataFrame([{
        "model_name": BEST_MODEL_NAME,
        "model_version": BEST_MODEL_VERSION,
        "annual_return": float(row["annualized_return_pct"].iloc[0]) / 100.0,
        "annual_return_pct": float(row["annualized_return_pct"].iloc[0]),
        "annual_vol": float(row["annualized_volatility_pct"].iloc[0]) / 100.0,
        "annual_vol_pct": float(row["annualized_volatility_pct"].iloc[0]),
        "sharpe": float(row["sharpe_ratio"].iloc[0]),
        "sortino": float(row["sortino_ratio"].iloc[0]),
        "max_drawdown": float(row["max_drawdown_pct"].iloc[0]) / 100.0,
        "max_drawdown_pct": float(row["max_drawdown_pct"].iloc[0]),
        "calmar": float(row["calmar_ratio"].iloc[0]),
        "worst_day": np.nan,
        "worst_day_pct": np.nan,
        "final_equity_multiple": float(row["ending_portfolio_value"].iloc[0]) / float(row["initial_capital"].iloc[0]),
        "ending_portfolio_value": float(row["ending_portfolio_value"].iloc[0]),
        "total_return_pct": float(row["total_return_pct"].iloc[0]),
        "number_of_trades": int(row["number_of_trades"].iloc[0]),
        "number_of_orders": int(row["number_of_orders"].iloc[0]),
        "win_rate_pct": float(row["win_rate_pct"].iloc[0]) if pd.notna(row["win_rate_pct"].iloc[0]) else np.nan,
        "profit_factor": float(row["profit_factor"].iloc[0]) if pd.notna(row["profit_factor"].iloc[0]) else np.nan,
        "start_date": str(row["backtest_start"].iloc[0]),
        "end_date": str(row["backtest_end"].iloc[0]),
    }])

elif "V12B_RL_GUARDIAN_DAILY" in globals():
    s = V12B_RL_GUARDIAN_DAILY.copy()
    if isinstance(s, pd.DataFrame):
        col = next((c for c in ["v12b_small_rl_guardian", "daily_return", "return", "ret"] if c in s.columns), s.select_dtypes(include=[np.number]).columns[0])
        date_col = next((c for c in ["date", "path_date", "timestamp_utc", "timestamp"] if c in s.columns), None)
        idx = pd.to_datetime(s[date_col], errors="coerce") if date_col else pd.to_datetime(s.index, errors="coerce")
        s = pd.Series(s[col].values, index=idx)
    else:
        s.index = pd.to_datetime(s.index, errors="coerce")
    summary = pd.DataFrame([_metric_from_daily(s)])
else:
    raise RuntimeError("No final backtest metrics or V12B_RL_GUARDIAN_DAILY found. Run the RL Guardian and backtest export cells first.")

display(summary)

if daily_path.exists():
    daily = pd.read_csv(daily_path)
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily["year"] = daily["date"].dt.year

    yearly = []
    for y, g in daily.groupby("year"):
        r = pd.Series(pd.to_numeric(g["daily_return"], errors="coerce").fillna(0.0).values, index=g["date"])
        m = _metric_from_daily(r)
        yearly.append({
            "year": int(y),
            "annual_return_pct": m["annual_return_pct"],
            "annual_vol_pct": m["annual_vol_pct"],
            "sharpe": m["sharpe"],
            "max_drawdown_pct": m["max_drawdown_pct"],
            "worst_day_pct": m["worst_day_pct"],
            "final_equity_multiple": m["final_equity_multiple"],
            "n_days": m["n_days"],
        })
    yearly_df = pd.DataFrame(yearly).sort_values("year")
    display(yearly_df)

    monthly = daily.copy()
    monthly["month"] = monthly["date"].dt.to_period("M").astype(str)
    monthly_df = monthly.groupby("month").agg(
        monthly_return_pct=("daily_return", lambda x: ((1.0 + pd.to_numeric(x, errors="coerce").fillna(0.0)).prod() - 1.0) * 100.0),
        worst_day_pct=("daily_return", lambda x: pd.to_numeric(x, errors="coerce").min() * 100.0),
        best_day_pct=("daily_return", lambda x: pd.to_numeric(x, errors="coerce").max() * 100.0),
        n_days=("daily_return", "size"),
    ).reset_index()
    display(monthly_df.tail(24))

if live_state_path.exists():
    with open(live_state_path, "r") as f:
        live_state = json.load(f)
    print("Live state:", live_state)

if artifact_meta_path.exists():
    with open(artifact_meta_path, "r") as f:
        meta = json.load(f)
    print("Artifact model:", meta.get("selected_model"))
    print("Artifact version:", meta.get("model_version"))

print("")
print("FINAL SELECTED MODEL:", BEST_MODEL_NAME)
print("MODEL VERSION:", BEST_MODEL_VERSION)
print("Backtest folder:", FINAL_BACKTEST_ROOT)
print("Artifact folder:", FINAL_ARTIFACT_ROOT)
print("")
print("Key numbers:")
print(f"Annual return: {summary['annual_return_pct'].iloc[0]:.2f}%")
print(f"Annual vol: {summary['annual_vol_pct'].iloc[0]:.2f}%")
print(f"Sharpe: {summary['sharpe'].iloc[0]:.2f}")
print(f"Max drawdown: {summary['max_drawdown_pct'].iloc[0]:.2f}%")
print(f"Final equity multiple: {summary['final_equity_multiple'].iloc[0]:.2f}x")
